# Data Preprocessing for Music Generation

This notebook demonstrates the preprocessing pipeline for converting MIDI files to piano-roll representations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import pretty_midi
from tqdm import tqdm
import matplotlib.pyplot as plt

# Add src to path
sys.path.append('../src')

from preprocessing.midi_parser import parse_midi_file
from preprocessing.piano_roll import midi_to_piano_roll, create_windows
from config import *

## 1. Load Dataset Metadata

In [ ]:
# Load MAESTRO dataset CSV
csv_path = os.path.join(RAW_MIDI_DIR, 'maestro-v3.0.0.csv')
df = pd.read_csv(csv_path)

# Filter for available files
df['basename'] = df['midi_filename'].apply(os.path.basename)
actual_files = [f for f in os.listdir(RAW_MIDI_DIR) if f.endswith('.midi')]
df = df[df['basename'].isin(actual_files)].copy()
df['midi_path'] = df['basename'].apply(lambda x: os.path.join(RAW_MIDI_DIR, x))

# Split by train/validation
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'validation'].copy()

print(f"Train files: {len(train_df)}")
print(f"Validation files: {len(val_df)}")
print(f"\nDataset info:")
print(df.head())

## 2. MIDI to Piano-Roll Conversion

In [ ]:
def midi_to_windows(midi_path, window_len=WINDOW_LEN, fs=FS, min_density=0.02):
    """
    Convert MIDI file to piano-roll windows.
    
    Args:
        midi_path: Path to MIDI file
        window_len: Length of each window (timesteps)
        fs: Sampling frequency (frames per second)
        min_density: Minimum note density to keep window
        
    Returns:
        List of piano-roll windows
    """
    try:
        # Load MIDI
        midi = pretty_midi.PrettyMIDI(midi_path)
        
        # Convert to piano-roll (88 keys: MIDI 21-108)
        pr = midi.get_piano_roll(fs=fs)[21:109, :]
        
        # Binarize
        pr = (pr > 0).astype(np.float32).T  # Shape: (time, 88)
        
        # Create windows
        windows = []
        for i in range(0, len(pr) - window_len + 1, window_len):
            w = pr[i:i+window_len]
            
            # Filter out silent windows
            if np.mean(w) >= min_density:
                windows.append(w)
        
        return windows
    except Exception as e:
        print(f"Error processing {midi_path}: {e}")
        return []

# Test on a single file
test_file = train_df['midi_path'].iloc[0]
test_windows = midi_to_windows(test_file)
print(f"\nTest file: {os.path.basename(test_file)}")
print(f"Generated {len(test_windows)} windows")
print(f"Window shape: {test_windows[0].shape if test_windows else 'N/A'}")

## 3. Visualize Piano-Roll

In [ ]:
if test_windows:
    # Plot first window
    plt.figure(figsize=(12, 6))
    plt.imshow(test_windows[0].T, aspect='auto', origin='lower', cmap='Blues')
    plt.xlabel('Time Steps')
    plt.ylabel('Piano Keys (88 keys)')
    plt.title('Piano-Roll Representation (Single Window)')
    plt.colorbar(label='Note Active')
    plt.tight_layout()
    plt.show()
    
    print(f"Note density: {np.mean(test_windows[0]):.3f}")
    print(f"Active notes: {np.sum(test_windows[0])} / {test_windows[0].size}")

## 4. Process All Files

In [ ]:
print("Processing training files...")
train_windows = []
for path in tqdm(train_df['midi_path'].tolist()):
    train_windows.extend(midi_to_windows(path))

print("\nProcessing validation files...")
val_windows = []
for path in tqdm(val_df['midi_path'].tolist()):
    val_windows.extend(midi_to_windows(path))

print(f"\nTotal training windows: {len(train_windows)}")
print(f"Total validation windows: {len(val_windows)}")

## 5. Save Processed Data

In [ ]:
# Convert to numpy arrays
train_data = np.stack(train_windows)
val_data = np.stack(val_windows)

print(f"Train data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")

# Save to disk
os.makedirs(PROCESSED_DIR, exist_ok=True)
np.save(TRAIN_PIANOROLL, train_data)
np.save(VAL_PIANOROLL, val_data)

print(f"\nSaved to:")
print(f"  - {TRAIN_PIANOROLL}")
print(f"  - {VAL_PIANOROLL}")

## 6. Data Statistics

In [ ]:
# Compute statistics
train_density = np.mean(train_data)
val_density = np.mean(val_data)

print("Dataset Statistics:")
print(f"  Train density: {train_density:.4f}")
print(f"  Val density: {val_density:.4f}")
print(f"  Train size: {train_data.nbytes / 1e6:.2f} MB")
print(f"  Val size: {val_data.nbytes / 1e6:.2f} MB")

# Plot density distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Density per window
train_densities = np.mean(train_data, axis=(1, 2))
ax1.hist(train_densities, bins=50, alpha=0.7, color='#4C72B0')
ax1.set_xlabel('Note Density')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Note Density (Train)')
ax1.grid(True, alpha=0.3)

# Active keys distribution
active_keys = np.sum(train_data, axis=1)  # Sum over time
key_usage = np.mean(active_keys, axis=0)  # Average over samples
ax2.plot(key_usage, color='#DD8452')
ax2.set_xlabel('Piano Key (0-87)')
ax2.set_ylabel('Average Usage')
ax2.set_title('Piano Key Usage Distribution')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

Preprocessing complete! The data is now ready for training:
- Piano-roll representation: 88 keys (MIDI 21-108)
- Window length: 128 timesteps
- Sampling rate: 16 Hz
- Binary representation (note on/off)
- Filtered for minimum note density